# Characterize nodules at their ground-truth locations with MedGemma

Runs `5b_characterize_ground_truth_nodules.py`: for every pylidc consensus nodule already recorded in `ground_truth_annotations.json` (patients 1-20), crops directly around that nodule's own real centroid/diameter (no detector involved) and asks MedGemma 1.5 to rate it on the same 9 pylidc attributes as `5_characterize_nodules.py` - then compares straight against the mean of that nodule's own radiologist annotations. No MONAI/simpleitk needed, since there's no detector step.

Before running anything:
1. **Runtime > Change runtime type > GPU** (T4 is fine).
2. Upload `lidc_idri_p1-20.zip` to your Google Drive (same zip used for the counting notebook - patients 1-20 + `annotations.csv`, ~1.1GB). Skip this if you already have it there from before.
3. Add a Colab secret named `HF_TOKEN` (key icon in the left sidebar) holding a Hugging Face access token that has accepted the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/freya-gul/rail.git
%cd rail
!git checkout medgemma-characterization-variants

Point this at wherever you uploaded `lidc_idri_p1-20.zip` in Drive. It unzips into `datasets/LDIC-IDRI-subset/` inside the cloned repo — that's the path this script's `DICOM_ROOT` already expects:

In [ ]:
ZIP_PATH = "/content/drive/MyDrive/lidc_idri_p1-20.zip"  # <-- update to your actual upload path
DATA_DIR = "datasets/LDIC-IDRI-subset"  # relative to the repo root (we've already %cd'd into rail)

import pathlib
assert pathlib.Path(ZIP_PATH).exists(), f"{ZIP_PATH} not found — check the path/upload"

In [ ]:
!mkdir -p {DATA_DIR}
!unzip -q {ZIP_PATH} -d {DATA_DIR}
!ls {DATA_DIR}

In [ ]:
# No monai/simpleitk needed - this script never touches the MONAI detector, only
# crops directly out of the raw DICOM series at the ground-truth nodule locations.
!pip install -q pydicom "transformers>=5.12.1" "huggingface_hub>=1.21.0"

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

Sanity check: GPU visible to torch (the script already defaults to `cuda` > `mps` > `cpu`):

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Run characterization

Prints live progress per nodule (predicted vs. ground-truth attribute values aren't shown inline, but pass/fail on JSON validation is) plus a running ETA. Resumable at the individual-nodule level — a killed run picks back up partway through a patient rather than redoing it.

In [ ]:
!python image_download/5b_characterize_ground_truth_nodules.py --start 1 --end 20

## Peek at partial results anytime

Run this whenever you want — while the run above is still going, if it got interrupted, or once it's fully done. It reads whatever `nodule_characteristics_gt/<patient>.json` files already exist on disk and computes the same MAE/bias summary the final cell below does, over however many nodules have actually been characterized so far. No need to wait for all `--end` patients to finish, and safe to re-run repeatedly as more come in.

In [ ]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "image_download")
from lidc_attributes import LIDC_ATTRIBUTES

OUTPUT_DIR = Path("image_download/nodule_characteristics_gt")
patient_files = sorted(OUTPUT_DIR.glob("LIDC-IDRI-*.json"))
rows = [row for f in patient_files for row in json.loads(f.read_text())]

print(f"{len(rows)} nodule(s) characterized so far across {len(patient_files)} patient(s)\n")
print(f"{'attribute':<18}{'MAE':>8}{'bias':>8}{'n':>6}   scale")
for a in LIDC_ATTRIBUTES:
    errs = [r[f"{a}_err"] for r in rows if r.get(f"{a}_err") is not None]
    mae = sum(abs(e) for e in errs) / len(errs) if errs else None
    bias = sum(errs) / len(errs) if errs else None
    lo, hi = min(LIDC_ATTRIBUTES[a]["labels"]), max(LIDC_ATTRIBUTES[a]["labels"])
    mae_str = f"{mae:.2f}" if mae is not None else "n/a"
    bias_str = f"{bias:+.2f}" if bias is not None else "n/a"
    print(f"{a:<18}{mae_str:>8}{bias_str:>8}{len(errs):>6}   {lo}-{hi}")

## Results

- `image_download/nodule_characteristics_gt/<patient>.json` — per-nodule predicted attributes, ground-truth mean, error, raw MedGemma response, and JSON-validation problems (if any).
- `image_download/characterize_ground_truth_comparison.csv` — the same data flattened across all patients, one row per nodule.
- `image_download/characterize_ground_truth_summary.json` — MAE and signed bias per attribute, aggregated across every nodule.

Quick look at the summary table:

In [ ]:
import json
summary = json.load(open("image_download/characterize_ground_truth_summary.json"))
for attr, stats in summary.items():
    print(f"{attr:<18} MAE={stats['mae']:.2f}  bias={stats['bias']:+.2f}  n={stats['n']}")

## Try prompt variants: anchored guidance / few-shot examples

Runs `5c_characterize_variants.py`, an A/B sibling of the script above. The first nodule characterized (`LIDC-IDRI-0001` #0) showed a near-worst-case spiculation miss (predicted 1 "No Spiculation" vs. ground truth 4.25 "Marked Spiculation") that matches a bias `bias_correction.json` already measured on the detector-based pipeline (subtlety -0.53, margin -0.55, spiculation -0.44). Two cheap levers to test before reaching for LoRA fine-tuning:

- `--anchored` — adds targeted anti-underrating guidance to the subtlety/margin/spiculation attribute descriptions. Free (a few extra sentences, no extra images).
- `--fewshot` — prepends two fixed calibration examples (real images + their consensus-rounded ground truth) as prior conversation turns: one unambiguous low-spiculation nodule and one unambiguous high-spiculation nodule, both 4/4-reader consensus. Costs roughly 2x the vision tokens/time per nodule.

Combinable, and each flag combination writes to its own directory (`nodule_characteristics_gt_anchored/`, `..._fewshot/`, `..._anchored_fewshot/`) so nothing clobbers the zero-shot baseline above. Try a small `--end` first (a handful of patients) before committing to the full 20 — few-shot in particular roughly doubles per-nodule runtime.

In [ ]:
!python image_download/5c_characterize_variants.py --anchored --start 1 --end 5

In [ ]:
!python image_download/5c_characterize_variants.py --fewshot --start 1 --end 5

In [ ]:
!python image_download/5c_characterize_variants.py --anchored --fewshot --start 1 --end 5

Compare all variants (plus the zero-shot baseline from above) side by side, over whatever patients each has completed so far — safe to re-run any time, doesn't require any of them to be finished:

In [ ]:
import importlib.util
from pathlib import Path

spec = importlib.util.spec_from_file_location("variants", "image_download/5c_characterize_variants.py")
variants_mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(variants_mod)

variants_mod.compare_variants({
    "baseline": Path("image_download/nodule_characteristics_gt"),
    "anchored": Path("image_download/nodule_characteristics_gt_anchored"),
    "fewshot": Path("image_download/nodule_characteristics_gt_fewshot"),
    "anchored+fewshot": Path("image_download/nodule_characteristics_gt_anchored_fewshot"),
})